In [ ]:
from lxml import objectify, etree
import pandas as pd
from dataclasses import dataclass

aircraftIdentifications = []
NS = {
    "nas": "http://www.faa.aero/nas/3.0",
    "base": "http://www.fixm.aero/base/3.0",
    "flight": "http://www.fixm.aero/flight/3.0",
    "foundation": "http://www.fixm.aero/foundation/3.0",
    "xsi": "http://www.w3.org/2001/XMLSchema-instance",
}

def safe_get(obj, attr_path, key=None, default=None):
    """
    Safely traverse nested attributes and call .get(key) at the end if key is provided.
    attr_path: list of attribute names to traverse.
    key: key to pass to .get() at the final attribute.
    default: value to return if any attribute is missing.
    """
    try:
        for attr in attr_path:
            obj = getattr(obj, attr, None)
            if obj is None:
                return default
        if key is not None and hasattr(obj, "get"):
            return obj.get(key)
        return obj
    except Exception:
        return default

def parse_message(doc):
    
    root = objectify.fromstring(doc.encode("utf-8")) # TODO: NEED TO DEBUG THIS

    # You still typically access by navigation/XPath:
    flight = root.xpath(
        ".//*[@xsi:type and contains(@xsi:type, 'NasFlightType')]",
        namespaces=NS
    )[0]
    aircraftIdentification = safe_get(flight, ["flightIdentification"], "aircraftIdentification")
    flightStatus = safe_get(flight, ["flightStatus"], "fdpsFlightStatus")
    departure = safe_get(flight, ["departure"])
    departurePoint = safe_get(departure, [], "departurePoint") if departure is not None else None
    depature_time = safe_get(departure, ["runwayPositionAndTime", "runwayTime", "actual"], "time")
    estimated_depature_time = safe_get(departure, ["runwayPositionAndTime", "runwayTime", "estimated"], "time")

    arrival = safe_get(flight, ["arrival"])
    arrivalPoint = safe_get(arrival, [], "arrivalPoint") if arrival is not None else None
    arrival_time = safe_get(arrival, ["runwayPositionAndTime", "runwayTime", "actual"], "time")
    estimated_arrival_time = safe_get(arrival, ["runwayPositionAndTime", "runwayTime", "estimated"], "time")

    # arrival_time = arrival.runwayPositionAndTime.runwayTime.actual.get("time")
    return (aircraftIdentification, flightStatus, departurePoint, depature_time, arrivalPoint, arrival_time)

def split_xml_documents(text: str):
    docs = []
    current = []

    for line in text.splitlines():
        if line.startswith("<?xml") and current:
            docs.append("\n".join(current))
            current = []
        current.append(line)

    if current:
        docs.append("\n".join(current))

    return docs

with open("../../data/interim/messages.log", "r", encoding="utf-8") as f:
    raw = f.read()

docs = split_xml_documents(raw)
sfdps_rows = []
for doc in docs:
    sfdps_row = parse_message(doc)
    sfdps_rows.append(sfdps_row)
df = pd.DataFrame(sfdps_rows, columns=[
    "aircraftIdentification",
    "flightStatus",
    "departurePoint",
    "departureTime",
    "arrivalPoint",
    "arrivalTime"
])
df